In [31]:
from pathlib import Path
import json
from sentence_transformers import SentenceTransformer
import numpy as np

In [32]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "offers_dataset.json"

with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

offers = data["offers"]
print(offers[0])

{'store': 'REWE', 'branch_region': None, 'product_name': 'Bio Kräuterseitlinge', 'brand': 'Rewe Bio', 'variant': None, 'offer_price': '2.49', 'currency': 'EUR', 'price_quantity_basis': None, 'pack_size_text': 'je 125-g-Schale', 'net_weight_g': None, 'drained_weight_g': None, 'unit_price': None, 'unit_price_basis': None, 'unit_price_text': '€ 19,92 / kg -', 'valid_from': None, 'valid_to': None, 'validity_text': '08.11. - 14.11.', 'requires_loyalty': None, 'loyalty_conditions': None, 'conditions_text': '€ 19,92 / kg - je 125-g-Schale', 'product_url': None, 'source_url': 'https://www.marktguru.de/r/rewe', 'fetched_at': '2026-11-10T17:39:39.006357+00:00', 'extraction_method': 'marktguru visible offer card', 'fetch_method': 'requests', 'verification': 'candidate: inspect source evidence', 'source_text': 'Brandneu Bio Kräuterseitlinge Marke: Rewe Bio Preis: € 2,49 Noch 4 Tage Gültig: 08.11. - 14.11. Händler: REWE € 19,92 / kg - je 125-g-Schale', 'section_context': None, 'value_origin': {'off

In [ ]:
def chunk_text(offers):
    documents = []

    for offer in offers:
        if offer.get("is_searchable") is False:
            continue

        parts = [offer["product_name"]]

        details = [
            ("Marke", offer.get("brand")),
            ("Variante", offer.get("variant")),
            ("Kategorie", offer.get("category")),
            ("Packung", offer.get("pack_size_text")),
            ("Händler", offer["store"]),
            ("Region", offer.get("branch_region")),
            ("Angegebener Preis", f'{offer["offer_price"]} {offer["currency"]}'),
            ("Grundpreis", offer.get("unit_price_text")),
            ("Gültigkeit laut Anzeige", offer.get("validity_text")),
            ("Kundenkartenbedingungen", offer.get("loyalty_conditions")),
            ("Weitere Hinweise", offer.get("conditions_text")),
        ]

        for label, value in details:
            # print(f"{label}: {value}")
            if value:
                parts.append(f"{label}: {value}")

        if offer.get("requires_loyalty") is True:
            parts.append("Kundenkarte erforderlich")

        documents.append({
            "record_id": offer["record_id"],
            "text": ". ".join(parts),
            "price": offer["offer_price"],
            "currency": offer["currency"],
            "price_quantity_basis": offer.get("price_quantity_basis"),
            "location": offer.get("branch_region"),
            "category": offer.get("category"),
            "store": offer["store"],
            "valid_from": offer.get("valid_from"),
            "valid_to": offer.get("valid_to"),
            "requires_loyalty": offer.get("requires_loyalty"),
            "source_url": offer["source_url"],
            "source_text": offer["source_text"],
        })

    return documents

document=chunk_text(offers)

In [ ]:
def do_chunking(documents, chunk_size=1200, overlap=200):
    if chunk_size <= 0 or not 0 <= overlap < chunk_size:
        raise ValueError("Use chunk_size > 0 and 0 <= overlap < chunk_size.")

    chunks = []

    for doc in documents:
        text = doc["text"].strip()
        start = 0
        chunk_number = 1

        while start < len(text):
            end = min(start + chunk_size, len(text))

            chunk = doc.copy()
            chunk["chunk_id"] = f"{doc['record_id']}::chunk-{chunk_number}"
            chunk["text"] = text[start:end]
            chunks.append(chunk)

            if end == len(text):
                break

            start = end - overlap
            chunk_number += 1

    return chunks


document_chunks = do_chunking(document, chunk_size=1200, overlap=200)
print(f"Created {len(document_chunks)} chunks from {len(document)} offers")

In [ ]:
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

In [ ]:
def do_embedding(chunks, model):
    if not chunks:
        raise ValueError("No offer chunks to embed.")

    texts = [f"passage: {chunk['text']}" for chunk in chunks]

    embeddings = model.encode(
        texts,
        batch_size=16,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    for chunk, embedding in zip(chunks, embeddings):
        chunk["embedding"] = embedding

    return embeddings


embeddings = do_embedding(document_chunks, embedding_model)
print("Embedding shape:", embeddings.shape)

In [ ]:
def save_chunks_embeddings(chunks, embeddings, output_dir, model_name):
    if embeddings.ndim != 2 or len(chunks) != embeddings.shape[0]:
        raise ValueError("Each chunk must have one embedding row.")

    saved_chunks = [
        {key: value for key, value in chunk.items() if key != "embedding"}
        for chunk in chunks
    ]

    config = {
        "embedding_model": model_name,
        "normalized": True,  # Matches normalize_embeddings=True in do_embedding()
        "chunk_count": len(saved_chunks),
        "embedding_dimension": int(embeddings.shape[1]),
    }

    output_dir.mkdir(parents=True, exist_ok=True)

    (output_dir / "chunks.json").write_text(
        json.dumps(saved_chunks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    np.save(output_dir / "embeddings.npy", embeddings, allow_pickle=False)
    (output_dir / "config.json").write_text(
        json.dumps(config, indent=2),
        encoding="utf-8",
    )

    print(f"Saved {len(saved_chunks)} chunks and embeddings to {output_dir}")


output_dir = PROJECT_ROOT / "data" / "processed" / "embeddings"

save_chunks_embeddings(
    document_chunks,
    embeddings,
    output_dir,
    model_name="intfloat/multilingual-e5-base",
)